# CSE556 Assignment 2: Mini Hindi-GPT (Advanced Notebook)

This notebook implements all assignment tasks end-to-end with an accuracy-focused training pipeline.

- Task 1: Data preprocessing + SentencePiece tokenizer (`vocab_size=5000`)
- Task 2: Decoder-only GPT-style LM training + perplexity + 50-token generation
- Task 3: GPT backbone classifier training + validation accuracy + 5 correct predictions export

Run cells top-to-bottom.

In [ ]:
# If needed, run once:
# %pip install -q torch sentencepiece pandas scikit-learn matplotlib tqdm

%load_ext autoreload
%autoreload 2

import os
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd
import sentencepiece as spm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
print("Device:", DEVICE, "| AMP:", USE_AMP)

ROOT = Path.cwd()
DATA_DIR = ROOT / "Data"
CORPUS_DIR = DATA_DIR / "hindi_corpus" / "train"
CLS_CSV = DATA_DIR / "text_classification_dataset" / "train.csv"

OUT = ROOT / "outputs"
(OUT / "checkpoints").mkdir(parents=True, exist_ok=True)
(OUT / "generations").mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "tokenizer").mkdir(parents=True, exist_ok=True)
(OUT / "splits").mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CORPUS exists:", CORPUS_DIR.exists())
print("CLS CSV exists:", CLS_CSV.exists())

In [ ]:
@dataclass
class CFG:
    # Tokenizer / sequence
    vocab_size: int = 5000
    block_size: int = 192

    # Model size (increase for better quality if GPU allows)
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1

    # LM training
    lm_batch_size: int = 24
    lm_epochs: int = 3
    lm_lr: float = 3e-4
    lm_warmup_steps: int = 200
    lm_weight_decay: float = 0.1
    lm_label_smoothing: float = 0.1
    grad_clip: float = 1.0

    # Classifier training
    cls_batch_size: int = 32
    cls_epochs: int = 12
    cls_lr: float = 8e-5
    cls_backbone_lr_mult: float = 0.2
    cls_weight_decay: float = 0.01
    early_stop_patience: int = 3

    # Data splits (90/10 for both tasks)
    val_ratio_lm: float = 0.1
    val_ratio_cls: float = 0.1

cfg = CFG()
cfg

In [ ]:
corpus_rows = []
for fp in tqdm(sorted(CORPUS_DIR.glob("*.txt")), desc="Reading corpus files"):
    try:
        txt = fp.read_text(encoding="utf-8", errors="ignore").strip()
        if txt:
            corpus_rows.append((fp.name, txt))
    except Exception:
        continue

print("Corpus docs:", len(corpus_rows))

ids = [r[0] for r in corpus_rows]
train_ids, val_ids = train_test_split(ids, test_size=cfg.val_ratio_lm, random_state=SEED)
train_id_set = set(train_ids)
val_id_set = set(val_ids)

corpus_train_texts = [t for fid, t in corpus_rows if fid in train_id_set]
corpus_val_texts = [t for fid, t in corpus_rows if fid in val_id_set]

(OUT / "splits" / "lm_train_ids.txt").write_text("\n".join(train_ids), encoding="utf-8")
(OUT / "splits" / "lm_val_ids.txt").write_text("\n".join(val_ids), encoding="utf-8")

print("LM train docs:", len(corpus_train_texts), "| LM val docs:", len(corpus_val_texts))
print("Leakage check (must be 0):", len(train_id_set.intersection(val_id_set)))

In [ ]:
# Corpus integrity audit: empty files are expected in this dataset and are skipped.
all_corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
empty_files = []
for fp in tqdm(all_corpus_files, desc="Auditing empty files"):
    try:
        txt = fp.read_text(encoding="utf-8", errors="ignore").strip()
        if not txt:
            empty_files.append(fp.name)
    except Exception:
        empty_files.append(fp.name)

empty_file_set = set(empty_files)
print("Total corpus files:", len(all_corpus_files))
print("Non-empty files used for LM:", len(corpus_rows))
print("Empty/unreadable files:", len(empty_files))
print("First 20 empty files:", empty_files[:20])

for fname in ["10784.txt", "134134.txt"]:
    print(f"{fname} empty?:", fname in empty_file_set)

In [ ]:
# Train SentencePiece tokenizer on LM train split only
sp_input = OUT / "tokenizer" / "sp_train_text.txt"
sp_input.write_text("\n".join(corpus_train_texts), encoding="utf-8")

sp_prefix = OUT / "tokenizer" / "hindi_sp_bpe"
sp_model = sp_prefix.with_suffix(".model")

if not sp_model.exists():
    spm.SentencePieceTrainer.train(
        input=str(sp_input),
        model_prefix=str(sp_prefix),
        vocab_size=cfg.vocab_size,
        model_type="bpe",
        character_coverage=0.9995,
        split_digits=False,
        unk_id=0,
        bos_id=1,
        eos_id=2,
        pad_id=3,
    )

sp = spm.SentencePieceProcessor(model_file=str(sp_model))

print("Tokenizer vocab:", sp.vocab_size())
print("Encode sample:", sp.encode(corpus_train_texts[0][:120], out_type=int)[:20])

In [ ]:
# Classification preprocessing
cls_df = pd.read_csv(CLS_CSV)
print("Columns:", list(cls_df.columns), "| rows:", len(cls_df))

TEXT_COL = "text" if "text" in cls_df.columns else cls_df.columns[0]
LABEL_COL = "experience" if "experience" in cls_df.columns else cls_df.columns[-1]

cls_df = cls_df[[TEXT_COL, LABEL_COL]].dropna().copy()

if cls_df[LABEL_COL].dtype == object:
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    cls_df["label_id"] = cls_df[LABEL_COL].astype(str).str.strip().str.lower().map(label_map)
else:
    cls_df["label_id"] = cls_df[LABEL_COL].astype(int)

cls_df = cls_df.dropna(subset=["label_id"]).copy()
cls_df["label_id"] = cls_df["label_id"].astype(int)

cls_train_df, cls_val_df = train_test_split(
    cls_df,
    test_size=cfg.val_ratio_cls,
    random_state=SEED,
    stratify=cls_df["label_id"],
)

print("Class distribution (full):")
print(cls_df["label_id"].value_counts().sort_index())
print("Cls train:", len(cls_train_df), "| Cls val:", len(cls_val_df))

In [ ]:
class LMDataset(Dataset):
    def __init__(self, texts: List[str], block_size: int, eos_id: int):
        self.block_size = block_size
        token_stream = []
        for t in texts:
            token_stream.extend(sp.encode(t, out_type=int))
            token_stream.append(eos_id)
        self.tokens = torch.tensor(token_stream, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.tokens) - self.block_size - 1)

    def __getitem__(self, idx):
        x = self.tokens[idx: idx + self.block_size]
        y = self.tokens[idx + 1: idx + self.block_size + 1]
        return x, y


class ClassificationDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, text_col: str, max_len: int, pad_id: int):
        self.texts = frame[text_col].astype(str).tolist()
        self.labels = frame["label_id"].astype(int).tolist()
        self.max_len = max_len
        self.pad_id = pad_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = sp.encode(self.texts[idx], out_type=int)[: self.max_len]
        if len(ids) < self.max_len:
            ids = ids + [self.pad_id] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


lm_train_ds = LMDataset(corpus_train_texts, cfg.block_size, eos_id=sp.eos_id())
lm_val_ds = LMDataset(corpus_val_texts, cfg.block_size, eos_id=sp.eos_id())
cls_train_ds = ClassificationDataset(cls_train_df, TEXT_COL, cfg.block_size, pad_id=sp.pad_id())
cls_val_ds = ClassificationDataset(cls_val_df, TEXT_COL, cfg.block_size, pad_id=sp.pad_id())

lm_train_loader = DataLoader(lm_train_ds, batch_size=cfg.lm_batch_size, shuffle=True, drop_last=True)
lm_val_loader = DataLoader(lm_val_ds, batch_size=cfg.lm_batch_size, shuffle=False, drop_last=True)
cls_train_loader = DataLoader(cls_train_ds, batch_size=cfg.cls_batch_size, shuffle=True)
cls_val_loader = DataLoader(cls_val_ds, batch_size=cfg.cls_batch_size, shuffle=False)

print("LM train batches:", len(lm_train_loader), "| LM val batches:", len(lm_val_loader))
print("CLS train batches:", len(cls_train_loader), "| CLS val batches:", len(cls_val_loader))

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, n_layer: int, n_head: int, n_embd: int, dropout: float):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            dim_feedforward=4 * n_embd,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
)
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=n_layer,
            norm=nn.LayerNorm(n_embd),
)

    def _causal_mask(self, T: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
        mask = torch.full((T, T), float("-inf"), device=device, dtype=dtype)
        return torch.triu(mask, diagonal=1)

    def forward(self, idx):
        _, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block size {self.block_size}")

        pos = torch.arange(0, T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)

        attn_mask = self._causal_mask(T, idx.device, x.dtype)
        x = self.transformer(x, mask=attn_mask)
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, backbone: MiniGPT, vocab_size: int):
        super().__init__()
        self.backbone = backbone
        self.lm_head = nn.Linear(backbone.token_emb.embedding_dim, vocab_size, bias=False)
        # Weight tying tends to improve LM quality for similar parameter budget.
        self.lm_head.weight = self.backbone.token_emb.weight

    def forward(self, idx, targets=None, label_smoothing=0.0):
        h = self.backbone(idx)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = nn.functional.cross_entropy(
                logits.reshape(B * T, V),
                targets.reshape(B * T),
                label_smoothing=label_smoothing,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=0.9, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.backbone.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


class GPTClassifier(nn.Module):
    def __init__(self, backbone: MiniGPT, pad_id: int, n_classes: int = 3):
        super().__init__()
        self.backbone = backbone
        self.pad_id = pad_id
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(backbone.token_emb.embedding_dim, n_classes)

    def forward(self, idx, targets=None):
        h = self.backbone(idx)
        valid_mask = (idx != self.pad_id).long()
        last_pos = (valid_mask.sum(dim=1) - 1).clamp(min=0)
        last_tok = h[torch.arange(h.size(0), device=h.device), last_pos]
        logits = self.fc(self.dropout(last_tok))
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(logits, targets)
        return logits, loss

In [ ]:
# Build LM model
backbone = MiniGPT(
    vocab_size=sp.vocab_size(),
    block_size=cfg.block_size,
    n_layer=cfg.n_layer,
    n_head=cfg.n_head,
    n_embd=cfg.n_embd,
    dropout=cfg.dropout,
).to(DEVICE)

lm_model = GPTLanguageModel(backbone, vocab_size=sp.vocab_size()).to(DEVICE)

lm_optimizer = torch.optim.AdamW(
    lm_model.parameters(),
    lr=cfg.lm_lr,
    betas=(0.9, 0.95),
    weight_decay=cfg.lm_weight_decay,
)

steps_per_epoch = max(1, len(lm_train_loader))
total_steps = steps_per_epoch * cfg.lm_epochs
lm_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    lm_optimizer,
    max_lr=cfg.lm_lr,
    total_steps=total_steps,
    pct_start=min(0.3, cfg.lm_warmup_steps / max(1, total_steps)),
    anneal_strategy="cos",
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("LM params:", sum(p.numel() for p in lm_model.parameters()) / 1e6, "M")
print("LM total steps:", total_steps)

In [ ]:
# LM training setup
best_ppl = float("inf")
lm_log = []
best_lm_path = OUT / "checkpoints" / "lm_best.pt"

print("Ready for LM training.")

In [ ]:
# LM training loop (run this cell after setup)
num_lm_epochs = cfg.lm_epochs
lm_train_losses = np.zeros(num_lm_epochs)
lm_val_losses = np.zeros(num_lm_epochs)
lm_train_ppls = np.zeros(num_lm_epochs)
lm_val_ppls = np.zeros(num_lm_epochs)

best_ppl = float("inf")
actual_lm_epochs = 0
lm_log = []

total_len = len(lm_train_loader) + len(lm_val_loader)

for epoch in range(num_lm_epochs):
    epoch_str = str(epoch + 1).rjust(len(str(num_lm_epochs)), " ")

    with tqdm(total=total_len, desc=f"LM Epoch [ {epoch_str}/{num_lm_epochs} ] : ") as pbar:
        lm_model.train()
        train_loss_list = []

        for x, y in lm_train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                _, loss = lm_model(x, y, label_smoothing=cfg.lm_label_smoothing)

            lm_optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(lm_optimizer)
            torch.nn.utils.clip_grad_norm_(lm_model.parameters(), cfg.grad_clip)
            scaler.step(lm_optimizer)
            scaler.update()
            lm_scheduler.step()

            train_loss_list.append(loss.item())
            pbar.update(1)

        train_loss = float(np.mean(train_loss_list)) if train_loss_list else float("inf")
        train_ppl = math.exp(train_loss) if train_loss < 20 else float("inf")

        lm_model.eval()
        val_loss_list = []
        with torch.no_grad():
            for x, y in lm_val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    _, vloss = lm_model(x, y, label_smoothing=0.0)

                val_loss_list.append(vloss.item())
                pbar.update(1)

        val_loss = float(np.mean(val_loss_list)) if val_loss_list else float("inf")
        val_ppl = math.exp(val_loss) if val_loss < 20 else float("inf")

        lm_train_losses[epoch] = train_loss
        lm_val_losses[epoch] = val_loss
        lm_train_ppls[epoch] = train_ppl
        lm_val_ppls[epoch] = val_ppl
        actual_lm_epochs += 1

        pbar.set_description(f"LM Epoch [ {epoch_str}/{num_lm_epochs} ]")
        pbar.set_postfix({
            "Train PPL": f"{train_ppl:.3f}",
            "Train loss": f"{train_loss:.3f}",
            "Val PPL": f"{val_ppl:.3f}",
            "Val loss": f"{val_loss:.3f}",
        })

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_ppl": train_ppl,
        "val_loss": val_loss,
        "val_ppl": val_ppl,
    }
    lm_log.append(row)
    print(row)

    if val_ppl < best_ppl:
        best_ppl = val_ppl
        torch.save(lm_model.state_dict(), best_lm_path)
        print(f"  [*] Best LM saved (Val PPL={val_ppl:.4f})")

print(f"\n[*] LM training complete. Best Val PPL = {best_ppl:.4f}")

In [ ]:
# LM training summary
print("Best val perplexity:", best_ppl)
print("Saved best LM checkpoint:", best_lm_path)
pd.DataFrame(lm_log)

In [ ]:
# Load best LM checkpoint before generation
lm_model.load_state_dict(torch.load(best_lm_path, map_location=DEVICE))

prompt = corpus_val_texts[0][:200] if len(corpus_val_texts) > 0 else "भारत एक"
prompt_ids = torch.tensor([sp.encode(prompt, out_type=int)], dtype=torch.long, device=DEVICE)

with torch.no_grad():
    out_ids = lm_model.generate(prompt_ids, max_new_tokens=50, temperature=0.85, top_k=50).squeeze(0).tolist()

generated_text = sp.decode(out_ids)
gen_file = OUT / "generations" / "generated_hindi.txt"
gen_file.write_text(generated_text, encoding="utf-8")

print("Prompt:\n", prompt)
print("\nGenerated:\n", generated_text[:700])
print("Saved generation:", gen_file)

In [ ]:
# Classifier: start from best LM backbone weights
lm_model.load_state_dict(torch.load(best_lm_path, map_location=DEVICE))
classifier = GPTClassifier(lm_model.backbone, pad_id=sp.pad_id(), n_classes=3).to(DEVICE)

# Class-balanced loss weights for small dataset
cls_counts = cls_train_df["label_id"].value_counts().sort_index()
weights = torch.tensor([1.0 / max(1, cls_counts.get(i, 1)) for i in range(3)], dtype=torch.float32, device=DEVICE)
weights = weights / weights.sum() * 3.0

# Differential learning rates: slower for pretrained backbone, faster for new head
backbone_params = list(classifier.backbone.parameters())
head_params = list(classifier.fc.parameters())
cls_optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": cfg.cls_lr * cfg.cls_backbone_lr_mult},
        {"params": head_params, "lr": cfg.cls_lr},
    ],
    weight_decay=cfg.cls_weight_decay,
)
cls_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

# Two-stage fine-tuning to improve stability: freeze lower blocks briefly, then unfreeze
for p in classifier.backbone.parameters():
    p.requires_grad = True
freeze_blocks = max(1, cfg.n_layer // 2)
for bi, block in enumerate(classifier.backbone.transformer.layers):
    if bi < freeze_blocks:
        for p in block.parameters():
            p.requires_grad = False

In [ ]:
# Classifier training setup
best_acc = -1.0
best_f1 = -1.0
patience = 0
cls_log = []
best_cls_path = OUT / "checkpoints" / "classifier_best.pt"

cls_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    cls_optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=True,
)

print("Ready for classifier training.")

In [ ]:
# Classifier training loop (run this cell after setup)
num_cls_epochs = cfg.cls_epochs
cls_train_losses = np.zeros(num_cls_epochs)
cls_val_losses = np.zeros(num_cls_epochs)
cls_train_f1s = np.zeros(num_cls_epochs)
cls_val_f1s = np.zeros(num_cls_epochs)
cls_train_accs = np.zeros(num_cls_epochs)
cls_val_accs = np.zeros(num_cls_epochs)

best_acc = -1.0
best_f1 = -1.0
patience = 0
actual_cls_epochs = 0
cls_log = []
best_y_true, best_y_pred = [], []

total_len = len(cls_train_loader) + len(cls_val_loader)

for epoch in range(num_cls_epochs):
    if epoch == 2:
        for p in classifier.backbone.parameters():
            p.requires_grad = True

    epoch_str = str(epoch + 1).rjust(len(str(num_cls_epochs)), " ")
    with tqdm(total=total_len, desc=f"CLS Epoch [ {epoch_str}/{num_cls_epochs} ] : ") as pbar:
        classifier.train()
        train_loss_list = []
        train_true, train_pred = [], []

        for x, y in cls_train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits, _ = classifier(x, None)
                loss = nn.functional.cross_entropy(logits, y, weight=weights)

            cls_optimizer.zero_grad(set_to_none=True)
            cls_scaler.scale(loss).backward()
            cls_scaler.unscale_(cls_optimizer)
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), cfg.grad_clip)
            cls_scaler.step(cls_optimizer)
            cls_scaler.update()

            train_loss_list.append(loss.item())
            train_pred.extend(logits.argmax(dim=-1).detach().cpu().tolist())
            train_true.extend(y.detach().cpu().tolist())
            pbar.update(1)

        train_loss = float(np.mean(train_loss_list)) if train_loss_list else float("inf")
        train_acc = accuracy_score(train_true, train_pred) if train_true else 0.0
        train_f1 = f1_score(train_true, train_pred, average="macro", zero_division=0) if train_true else 0.0

        classifier.eval()
        val_loss_list = []
        y_true, y_pred = [], []

        with torch.no_grad():
            for x, y in cls_val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    logits, _ = classifier(x, None)
                    vloss = nn.functional.cross_entropy(logits, y, weight=weights)

                val_loss_list.append(vloss.item())
                y_pred.extend(logits.argmax(dim=-1).cpu().tolist())
                y_true.extend(y.cpu().tolist())
                pbar.update(1)

        val_loss = float(np.mean(val_loss_list)) if val_loss_list else float("inf")
        val_acc = accuracy_score(y_true, y_pred) if y_true else 0.0
        val_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0) if y_true else 0.0

        cls_train_losses[epoch] = train_loss
        cls_val_losses[epoch] = val_loss
        cls_train_f1s[epoch] = train_f1
        cls_val_f1s[epoch] = val_f1
        cls_train_accs[epoch] = train_acc
        cls_val_accs[epoch] = val_acc
        actual_cls_epochs += 1

        pbar.set_description(f"CLS Epoch [ {epoch_str}/{num_cls_epochs} ]")
        pbar.set_postfix({
            "Train F1": f"{train_f1:.3f}",
            "Train loss": f"{train_loss:.3f}",
            "Val F1": f"{val_f1:.3f}",
            "Val loss": f"{val_loss:.3f}",
        })

    cls_scheduler.step(val_f1)

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
    }
    cls_log.append(row)
    print(row)

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_acc = max(best_acc, val_acc)
        patience = 0
        best_y_true, best_y_pred = list(y_true), list(y_pred)
        torch.save(classifier.state_dict(), best_cls_path)
        print(f"  [*] Best model saved (F1={val_f1:.4f}, ACC={val_acc:.4f})")
    else:
        best_acc = max(best_acc, val_acc)
        patience += 1
        if patience >= cfg.early_stop_patience:
            print(f"  [!] Early stopping triggered at epoch {epoch + 1}")
            break

if not best_y_true:
    best_y_true, best_y_pred = list(y_true), list(y_pred)
y_true, y_pred = best_y_true, best_y_pred

print(f"\n[*] Classifier training complete. Best Val F1 = {best_f1:.4f}")

In [ ]:
# Classifier training summary
print("Best validation F1:", best_f1)
print("Best validation accuracy:", best_acc)
print("Saved best classifier checkpoint:", best_cls_path)
print("\nValidation Classification Report (best epoch):")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
pd.DataFrame(cls_log)

In [ ]:
# Save 5 correctly predicted validation samples
classifier.load_state_dict(torch.load(best_cls_path, map_location=DEVICE))
classifier.eval()

correct_rows = []
with torch.no_grad():
    for i in range(len(cls_val_ds)):
        x, y = cls_val_ds[i]
        logits, _ = classifier(x.unsqueeze(0).to(DEVICE), None)
        pred = int(logits.argmax(dim=-1).item())
        gold = int(y.item())
        if pred == gold:
            text = str(cls_val_df.iloc[i][TEXT_COL])
            correct_rows.append((text, pred, gold))
        if len(correct_rows) >= 5:
            break

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
correct_file = OUT / "predictions" / "5_correct_samples.txt"
with correct_file.open("w", encoding="utf-8") as f:
    for j, (text, pred, gold) in enumerate(correct_rows, 1):
        f.write(f"Sample {j}\n")
        f.write(f"Predicted: {id2label[pred]} ({pred})\n")
        f.write(f"Gold: {id2label[gold]} ({gold})\n")
        f.write("Text:\n")
        f.write(text.strip() + "\n")
        f.write("-" * 80 + "\n")

print("Saved correct-sample file:", correct_file)
print("Count saved:", len(correct_rows))

In [ ]:
lm_df = pd.DataFrame(lm_log)
cls_df_log = pd.DataFrame(cls_log)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if len(lm_df) > 0:
    ax[0].plot(lm_df["epoch"], lm_df["val_ppl"], marker="o")
    ax[0].set_title("LM Validation Perplexity")
    ax[0].set_xlabel("Epoch")
    ax[0].set_ylabel("Perplexity")
    ax[0].grid(True)

if len(cls_df_log) > 0:
    ax[1].plot(cls_df_log["epoch"], cls_df_log["val_acc"], marker="o")
    ax[1].set_title("Classifier Validation Accuracy")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Accuracy")
    ax[1].grid(True)

plt.tight_layout()
plt.show()

summary = {
    "best_val_ppl": float(min([r["val_ppl"] for r in lm_log])) if lm_log else None,
    "best_val_acc": float(max([r["val_acc"] for r in cls_log])) if cls_log else None,
    "lm_ckpt": str(best_lm_path),
    "cls_ckpt": str(best_cls_path),
    "generation_file": str(gen_file),
    "correct_samples_file": str(correct_file),
}
summary

## High-Performance Tuning Knobs (Optional)

To push performance higher, increase these in `CFG` if compute allows:

- `block_size`: `192 -> 256/384`
- `n_layer`: `6 -> 8/10`
- `n_embd`: `384 -> 512`
- `lm_epochs`: `3 -> 6+`
- Lower `cls_lr` slightly (`8e-5 -> 5e-5`) for stable fine-tuning

Suggested workflow:
1. Train LM until val perplexity plateaus.
2. Re-run classifier from best LM checkpoint.
3. Keep best run by val accuracy and preserve corresponding checkpoints.